# Play Store — 14 SQL questions

Cleaned app catalogue + review sentiment. Pandas first, then the same
queries in SQLite.

Ratings that were filled with the column mean have been put back to missing.


In [1]:
from pathlib import Path
import sqlite3
import pandas as pd
import matplotlib.pyplot as plt

ROOT = None
for cand in [Path(".."), Path("."), Path("/workspace/artifacts/playstore-apps-analysis")]:
    if (cand / "data" / "playstore_apps.csv").exists():
        ROOT = cand
        break
DATA, FIG = ROOT / "data", ROOT / "reports" / "figures"
FIG.mkdir(parents=True, exist_ok=True)
apps = pd.read_csv(DATA / "playstore_apps.csv")
revs = pd.read_csv(DATA / "playstore_reviews.csv")
print(apps.shape, revs.shape, "rating NA", apps.rating.isna().sum())


(9648, 13) (29692, 5) rating NA 1458


## Q1–Q3 Rating and reviews

In [2]:
mx = apps["rating"].max()
top = (apps[apps.rating == mx][["app", "rating", "reviews", "installs"]]
       .sort_values("reviews", ascending=False))
print("apps at max rating", mx, ":", len(top))
display(top.head(8))
print("most reviews overall:")
display(apps.nlargest(1, "reviews")[["app", "category", "reviews", "installs"]])


apps at max rating 5.0 : 271


,app,rating,reviews,installs
9174,Ríos de Fe,5.0,141,1000
9120,"FD Calculator (EMI, SIP, RD & Loan Eligilibility)",5.0,104,1000
6984,Oración CX,5.0,103,5000
5786,Barisal University App-BU Face,5.0,100,1000
8360,Master E.K,5.0,90,1000
6448,CL REPL,5.0,47,1000
4272,AJ Cam,5.0,44,100
8381,Ek Vote,5.0,43,500


most reviews overall:


,app,category,reviews,installs
2000,Facebook,SOCIAL,78158306,1000000000


## Q4–Q6 Revenue, category installs, genre count

In [3]:
paid = apps[apps.type == "Paid"].copy()
paid["revenue"] = paid.price * paid.installs
print("paid revenue", round(paid.revenue.sum(), 2))
display(paid.nlargest(5, "revenue")[["app", "price", "installs", "revenue"]])
cat = apps.groupby("category")["installs"].sum().sort_values(ascending=False)
display(cat.head(5).to_frame("installs"))
display(apps.genres.value_counts().head(5).to_frame("n"))


paid revenue 291097457.89


,app,price,installs,revenue
1741,Minecraft,6.99,10000000,69900000.0
4392,I am rich,399.99,100000,39999000.0
4396,I Am Rich Premium,399.99,50000,19999500.0
3206,Hitman Sniper,0.99,10000000,9900000.0
6362,Grand Theft Auto: San Andreas,6.99,1000000,6990000.0


,installs
category,
GAME,13878924415
COMMUNICATION,11038276251
TOOLS,8001271905
PRODUCTIVITY,5793091369
SOCIAL,5487867902


,n
genres,
Tools,824
Entertainment,560
Education,509
Business,420
Medical,395


## Q7–Q10 Games, Android version, free/paid, dating

In [4]:
display(apps[apps.category == "GAME"]
        .sort_values("installs", ascending=False)
        [["app", "installs", "reviews"]].head(8))
print("android 4.0.3 and up", int((apps.android_ver == "4.0.3 and up").sum()))
print(apps.type.value_counts().to_dict())
display(apps[apps.category == "DATING"]
        .nlargest(3, "reviews")[["app", "reviews", "rating", "installs"]])


,app,installs,reviews
1354,Subway Surfers,1000000000,27722264
1355,Candy Crush Saga,500000000,22426677
1408,My Talking Tom,500000000,14891223
1362,Pou,500000000,10485308
1361,Temple Run 2,500000000,8118609
1353,ROBLOX,100000000,4447388
1380,Angry Birds Classic,100000000,5566669
1406,Garena Free Fire,100000000,5465624


android 4.0.3 and up 1395
{'Free': 8895, 'Paid': 753}


,app,reviews,rating,installs
411,Zoosk Dating App: Meet Singles,516801,4.0,10000000
418,"Moco - Chat, Meet People",313724,4.2,10000000
420,Hot or Not - Find someone right now,305708,4.1,10000000


## Q11–Q14 Reviews

In [5]:
food = revs[revs.app == "10 Best Foods for You"]
print("10 Best Foods", food.sentiment.value_counts().to_dict())
asus = revs[(revs.app == "ASUS SuperNote") & (revs.sentiment_polarity == 1)
            & (revs.sentiment_subjectivity == 1)]
print("ASUS", asus.translated_review.tolist())
abs_n = revs[(revs.app == "Abs Training-Burn belly fat") & (revs.sentiment == "Neutral")]
print("Abs Training neutral", len(abs_n))
adobe = revs[(revs.app == "Adobe Acrobat Reader") & (revs.sentiment == "Negative")]
print("Adobe negative", len(adobe))


10 Best Foods {'Positive': 79, 'Neutral': 11, 'Negative': 5}
ASUS ['Awesome!!!!']
Abs Training neutral 5
Adobe negative 20


## Same headlines in SQLite

In [6]:
con = sqlite3.connect(":memory:")
apps.to_sql("apps", con, index=False, if_exists="replace")
revs.to_sql("reviews", con, index=False, if_exists="replace")
print("Facebook reviews SQL", pd.read_sql(
    "SELECT app, reviews FROM apps ORDER BY reviews DESC LIMIT 1", con).to_dict("records"))
print("GAME installs SQL", pd.read_sql(
    "SELECT category, SUM(installs) AS n FROM apps GROUP BY category ORDER BY n DESC LIMIT 1", con).to_dict("records"))
print("Tools genre SQL", pd.read_sql(
    "SELECT genres, COUNT(*) n FROM apps GROUP BY genres ORDER BY n DESC LIMIT 1", con).to_dict("records"))
print("Zoosk SQL", pd.read_sql(
    "SELECT app, reviews FROM apps WHERE category='DATING' ORDER BY reviews DESC LIMIT 1", con).to_dict("records"))
print("Food sentiment SQL")
display(pd.read_sql(
    "SELECT sentiment, COUNT(*) n FROM reviews WHERE app='10 Best Foods for You' GROUP BY sentiment", con))
con.close()


Facebook reviews SQL [{'app': 'Facebook', 'reviews': 78158306}]
GAME installs SQL [{'category': 'GAME', 'n': 13878924415}]
Tools genre SQL [{'genres': 'Tools', 'n': 824}]
Zoosk SQL [{'app': 'Zoosk Dating App: Meet Singles', 'reviews': 516801}]
Food sentiment SQL


,sentiment,n
0,Negative,5
1,Neutral,11
2,Positive,79


## Takeaways

- 5.0 is common and small-sample. Use reviews, not the star cap.
- GAME leads installs; Facebook leads reviews.
- Paid revenue is a price × bucket estimate ($291.1M), not receipts.
